# APIGamings - Test Notebook

This notebook contains tests for the game API functions.

## Install required packages first
```
%pip install python-dotenv transformers torch datasets
```

In [ ]:
# Import Hugging Face libraries
import transformers
import torch
from datasets import load_dataset
import datasets

print("Hugging Face libraries imported successfully!")
print(f"Transformers version: {transformers.__version__}")
print(f"Torch version: {torch.__version__}")
print(f"Datasets version: {datasets.__version__}")

# Test loading a dataset
try:
    dataset = load_dataset("imdb")
    print(f"IMDB dataset loaded successfully! Shape: {dataset['train'].shape}")
except Exception as e:
    print(f"Error loading dataset: {e}")

In [ ]:
import requests
import json
import os

In [ ]:
# Função que verifica existência de cache, chama dados e salva

def get_games():
    """Função completa com cache"""
    cache_json = "cache.json"
    
    # Verifica se cache existe
    if os.path.exists(cache_json):
        print("Using cache")
        with open(cache_json, "r") as f:
            return json.load(f)
    
    # GET request if cache doesn't exist
    print("Fetching from API")
    url = "https://www.freetogame.com/api/games"
    response = requests.get(url)
    data = response.json()
    
    # Save to file
    with open(cache_json, "w") as f:
        json.dump(data, f, indent=2)
    
    print(f"Data saved in {cache_json}")
    return data

# App usage
games = get_games()
print(f"Total games: {len(games)}")

In [ ]:
def get_all_games():
    endpoint = "https://www.freetogame.com/api/games"
    response = requests.get(endpoint)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code}")
        return None

# Testar a função
games = get_all_games()
if games:
    print(f"Total games found: {len(games)}")
    print(f"First game: {games[0]['title']}")
else:
    print("Failed to fetch games")

## Test 2: Extract unique genres

In [ ]:
def get_all_genres():
    games = get_all_games()
    if not games:
        return []
    
    genres = set()
    for game in games:
        genres.add(game['genre'])
    return sorted(list(genres))

# Testar a função
genres = get_all_genres()
print(f"Total genres: {len(genres)}")
print("Available genres:")
for i, genre in enumerate(genres[:10], 1):
    print(f"{i}. {genre}")
if len(genres) > 10:
    print(f"... and {len(genres) - 10} more genres")

## Test 3: Filter games by genre

In [ ]:
def filter_games_by_genre(genre):
    games = get_all_games()
    if not games:
        return []
    
    game_filters = [
        game for game in games 
        if genre.lower() in game['genre'].lower()
    ]
    return game_filters

# Testar com diferentes gêneros
test_genres = ["Shooter", "MMO", "RPG"]

for genre in test_genres:
    filtered_games = filter_games_by_genre(genre)
    print(f"\nGenre: {genre}")
    print(f"Games found: {len(filtered_games)}")
    if filtered_games:
        print(f"Example: {filtered_games[0]['title']}")

## Test 4: Search specific game by ID

In [ ]:
def search_games(genre, game_id):
    endpoint = f"https://www.freetogame.com/api/games?genre={genre}&id={game_id}"
    response = requests.get(endpoint)
    games_data = response.json() if response.status_code == 200 else None
    return games_data

# Testar busca por ID
test_id = "1"
result = search_games("", test_id)

if result:
    print(f"Game found (ID {test_id}):")
    print(f"Title: {result[0]['title']}")
    print(f"Genre: {result[0]['genre']}")
    print(f"Platform: {result[0]['platform']}")
else:
    print(f"No game found with ID {test_id}")

## Test 5: Data Analysis

In [ ]:
# Análise dos dados
games = get_all_games()
if games:
    # Estatísticas básicas
    platforms = {}
    genres_count = {}
    
    for game in games:
        # Contar plataformas
        platform = game.get('platform', 'Unknown')
        platforms[platform] = platforms.get(platform, 0) + 1
        
        # Contar gêneros
        genre = game.get('genre', 'Unknown')
        genres_count[genre] = genres_count.get(genre, 0) + 1
    
    print("=== GAME STATISTICS ===")
    print(f"Total games: {len(games)}")
    print("\nPlatforms:")
    for platform, count in sorted(platforms.items(), key=lambda x: x[1], reverse=True):
        print(f"  {platform}: {count} games")
    
    print("\nTop 5 Genres:")
    for genre, count in sorted(genres_count.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {genre}: {count} games")

## Test 7: Data Cache

In [ ]:
import os

def get_games_with_cache():
    """Função que verifica existência de cache, chama dados e salva"""
    cache_json = "cache.json"
    
    # Verifica se cache existe
    if os.path.exists(cache_json):
        print("Using cache")
        with open(cache_json, "r") as f:
            data = json.load(f)
        return data
    
    # If not exists, make request
    print("Fetching from API")
    url = "https://www.freetogame.com/api/games"
    response = requests.get(url)
    data = response.json()
    
    # Save to cache
    with open(cache_json, "w") as f:
        json.dump(data, f, indent=2)
    
    print(f"Data saved in {cache_json}")
    return data

# Testar função de cache
games_cached = get_games_with_cache()
if games_cached:
    print(f"Total games in cache: {len(games_cached)}")
    print("Cache working!")
else:
    print("Cache error")

## Test 8: Environment Variables Test

In [ ]:
# Testar leitura das chaves do .env
def test_env_variables():
    """Testa se as variáveis de ambiente estão sendo lidas corretamente"""
    
    # Chaves esperadas no .env
    expected_keys = [
        'TWITCH_CLIENT_ID',
        'TWITCH_CLIENT_SECRET',
        'TWITCH_REDIRECT_URI'
    ]
    
    print("=== TESTE DE VARIÁVEIS DE AMBIENTE ===")
    
    # Verificar cada chave
    for key in expected_keys:
        value = os.getenv(key)
        if value:
            print(f"  {key}: {value[:10]}...***")  # Mostra apenas os primeiros 10 caracteres para segurança
        else:
            print(f"  {key}: NÃO ENCONTRADA")
    
    # Verificar se arquivo .env existe
    env_file = '.env'
    if os.path.exists(env_file):
        print(f"\n  Arquivo .env encontrado em: {os.path.abspath(env_file)}")
    else:
        print(f"\n  Arquivo .env NÃO encontrado em: {os.path.abspath(env_file)}")
        print("  Dica: Crie um arquivo .env com as chaves TWITCH_CLIENT_ID e TWITCH_CLIENT_SECRET")
    
    print("\n" + "="*50)

# Executar o teste
test_env_variables()